# Practical 01 — Feature Engineering

Execute each code cell with Shift + Enter. Preprocessing is fitted on training data only to prevent data leakage.

## 1. Import libraries

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline


## 2. Load the Titanic dataset from OpenML

In [ ]:
data = fetch_openml("titanic", version=1, as_frame=True)
df = data.frame.copy()
df = df.drop(columns=["boat", "body", "home.dest"], errors="ignore")
df.head()


## 3. Inspect missing values and data types

In [ ]:
print("Dataset shape:", df.shape)
display(df.isna().sum().sort_values(ascending=False).head(15))
display(df.dtypes)


## 4. Separate features and target

In [ ]:
y = df["survived"].astype(int)
X = df.drop(columns=["survived"])
X = X.drop(columns=["name", "ticket", "cabin"], errors="ignore")
numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()
print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)


## 5. Split before fitting preprocessing

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


## 6. Impute missing values, encode categories and scale numbers

In [ ]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])
X_train_ready = preprocessor.fit_transform(X_train)
X_test_ready = preprocessor.transform(X_test)
print("Before preprocessing:", X_train.shape)
print("After preprocessing:", X_train_ready.shape)


## 7. Verify model-ready data

In [ ]:
feature_names = preprocessor.get_feature_names_out()
values = X_train_ready.toarray() if hasattr(X_train_ready, "toarray") else X_train_ready
processed_df = pd.DataFrame(values, columns=feature_names)
display(processed_df.head())
print("Missing values after preprocessing:", np.isnan(processed_df.to_numpy()).sum())
